# DATATHON Terra Journey - Grupo 48 

## Preparação da Base de Dados para Análise

Este notebook apresenta o processo de preparação da base de dados utilizada no desenvolvimento do projeto Datathon. 

O objetivo desta etapa é integrar, padronizar e tratar os dados provenientes das diferentes bases anuais 2022, 2023 e 2024, garantindo sua qualidade e consistência para as etapas subsequentes de análise.

Durante o processamento são realizadas as seguintes atividades:

- Leitura das planilhas de dados;
- Análise da estrutura das bases e identificação dos atributos;
- Padronização dos nomes das variáveis;
- Conversão e adequação dos tipos de dados;
- Tratamento de inconsistências nos campos de idade e ano de nascimento;
- Remoção de atributos com baixa disponibilidade de informações;
- Consolidação das bases em um único conjunto de dados;
- Organização da estrutura final dos atributos;
- Exportação da base tratada para utilização nas etapas de Análise Exploratória de Dados (EDA) e modelagem.

Ao final deste processo, obtém-se uma base de dados unificada, consistente e preparada para suportar as análises estatísticas, a construção de modelos preditivos e a geração de insights para o projeto.

## 1. Importação das bibliotecas

Importação das bibliotecas necessárias para leitura, tratamento e manipulação dos dados.

In [1]:
import pandas as pd
import numpy as np
from collections import defaultdict 


## 2. Definição das funções auxiliares

Nesta etapa são definidas todas as funções utilizadas ao longo do processo, incluindo:

- leitura das planilhas;
- validação das colunas;
- remoção de atributos com poucos dados;
- conversão de tipos de dados;
- exportação do arquivo final.

In [2]:
def ler_planilhas_excel_xlsx(github):
    excel = pd.ExcelFile(github, engine='openpyxl')
    
    planilhas = {}
    info_abas = {}
    
    for aba in excel.sheet_names:
        
        df = pd.read_excel(excel, sheet_name=aba)
        planilhas[aba] = df
        
        info_abas[aba] = { 
            'nome_aba': aba,
            'num_linhas': len(df),
            'num_colunas': len(df.columns),
            'nome_colunas': list(df.columns),
            'tipos': {coluna: str(df[coluna].dtype) for coluna in df.columns},
            'nulos': {coluna: df[coluna].isnull().sum() for coluna in df.columns}
        }

    return planilhas, info_abas


In [3]:
def validar_colunas(dict_info_abas):
    grupos = defaultdict(list)
    
    for aba, info in info_abas.items():
        colunas = tuple(sorted(info['nome_colunas']))
        grupos[colunas].append(aba)
        
    for i, (colunas, abas) in enumerate(grupos.items(), start=1):
        print(f"\nGrupo {i}")
        print(f"Abas: {abas}")
        print(f"Quantidade de colunas: {len(colunas)}")
        print("Colunas:")
        for coluna in colunas:
            print(f" - {coluna}")

    grupos_lista = list(grupos.items())

    for i in range(len(grupos_lista)):
        colunas_i, abas_i = grupos_lista[i]
        
        for j in range(i + 1, len(grupos_lista)):
            colunas_j, abas_j = grupos_lista[j]
            
            set_i = set(colunas_i)
            set_j = set(colunas_j)
            
            print(f"\nComparação: Grupo {i+1} x Grupo {j+1}")
            print(f"Abas grupo {i+1}: {abas_i}")
            print(f"Abas grupo {j+1}: {abas_j}")
            print(f"Só no grupo {i+1}: {sorted(set_i - set_j)}")
            print(f"Só no grupo {j+1}: {sorted(set_j - set_i)}")

In [4]:
def remover_colunas_poucos_dados(df, minimo_non_null):

    colunas_remover = df.columns[df.count() < minimo_non_null]

    print(f'{len(colunas_remover)} colunas removidas:')
    print(colunas_remover.tolist())

    return df.drop(columns=colunas_remover)

In [5]:
def converter_campo(df, coluna, tipo):

    def converter_valor(valor):

        if pd.isna(valor):

            return pd.NA

        if tipo == "ano":

            data = pd.to_datetime(valor, errors="coerce")

            if pd.isna(data):

                return pd.NA

            if (

                data.year == 1970

                and data.month == 1

                and data.day == 1

            ):

                return data.microsecond * 1000 + data.nanosecond

            return data.year

        if tipo == "idade":

            if isinstance(valor, pd.Timestamp):

                if valor.year == 1900 and valor.month == 1:

                    return valor.day

            texto = str(valor).strip()

            # Trata strings como "1900-01-08 00:00:00"

            data = pd.to_datetime(texto, errors="coerce")

            if pd.notna(data) and data.year == 1900 and data.month == 1:

                return data.day

            # Trata números como 8, 9, "7"

            numero = pd.to_numeric(valor, errors="coerce")

            if pd.notna(numero):

                return int(numero)

            return pd.NA

    df[coluna] = df[coluna].apply(converter_valor).astype("Int64")

In [6]:
def salvar_excel(df, nome_arquivo):
    df.to_excel(
        nome_arquivo,
        index=False,
        engine='openpyxl'
    )
    print(f'Arquivo salvo com sucesso: {nome_arquivo}')

## 3. Leitura da base de dados

Leitura do arquivo Excel hospedado no GitHub contendo as bases dos diferentes anos do projeto PEDE.

In [7]:
github =  'https://raw.githubusercontent.com/andersonserrico/Datathon_TerraJourney_Grupo48/main/dados/BASE_DATATHON_2024.xlsx'

planilhas, info_abas = ler_planilhas_excel_xlsx(github)
validar_colunas(info_abas)


Grupo 1
Abas: ['PEDE2022']
Quantidade de colunas: 42
Colunas:
 - Ano ingresso
 - Ano nasc
 - Atingiu PV
 - Avaliador1
 - Avaliador2
 - Avaliador3
 - Avaliador4
 - Cf
 - Cg
 - Ct
 - Defas
 - Destaque IDA
 - Destaque IEG
 - Destaque IPV
 - Fase
 - Fase ideal
 - Gênero
 - IAA
 - IAN
 - IDA
 - IEG
 - INDE 22
 - IPS
 - IPV
 - Idade 22
 - Indicado
 - Inglês
 - Instituição de ensino
 - Matem
 - Nome
 - Nº Av
 - Pedra 20
 - Pedra 21
 - Pedra 22
 - Portug
 - RA
 - Rec Av1
 - Rec Av2
 - Rec Av3
 - Rec Av4
 - Rec Psicologia
 - Turma

Grupo 2
Abas: ['PEDE2023']
Quantidade de colunas: 48
Colunas:
 - Ano ingresso
 - Atingiu PV
 - Avaliador1
 - Avaliador2
 - Avaliador3
 - Avaliador4
 - Cf
 - Cg
 - Ct
 - Data de Nasc
 - Defasagem
 - Destaque IDA
 - Destaque IEG
 - Destaque IPV
 - Destaque IPV.1
 - Fase
 - Fase Ideal
 - Gênero
 - IAA
 - IAN
 - IDA
 - IEG
 - INDE 2023
 - INDE 22
 - INDE 23
 - IPP
 - IPS
 - IPV
 - Idade
 - Indicado
 - Ing
 - Instituição de ensino
 - Mat
 - Nome Anonimizado
 - Nº Av
 - P

## 4. Validação das estruturas das planilhas

Inspeção inicial das planilhas referentes aos anos de 2022, 2023 e 2024 para identificar diferenças estruturais e necessidades de padronização.

Verificação da estrutura de cada aba, incluindo:

- quantidade de linhas;
- quantidade de colunas;
- nomes das colunas;
- tipos dos dados;
- quantidade de valores nulos.

In [8]:
planilhas['PEDE2022'].info(10)

<class 'pandas.DataFrame'>
RangeIndex: 860 entries, 0 to 859
Data columns (total 42 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   RA                     860 non-null    str    
 1   Fase                   860 non-null    int64  
 2   Turma                  860 non-null    str    
 3   Nome                   860 non-null    str    
 4   Ano nasc               860 non-null    int64  
 5   Idade 22               860 non-null    int64  
 6   Gênero                 860 non-null    str    
 7   Ano ingresso           860 non-null    int64  
 8   Instituição de ensino  860 non-null    str    
 9   Pedra 20               323 non-null    str    
 10  Pedra 21               462 non-null    str    
 11  Pedra 22               860 non-null    str    
 12  INDE 22                860 non-null    float64
 13  Cg                     860 non-null    int64  
 14  Cf                     860 non-null    int64  
 15  Ct               

In [9]:
planilhas['PEDE2022'].head()

,RA,Fase,Turma,Nome,Ano nasc,Idade 22,Gênero,Ano ingresso,Instituição de ensino,Pedra 20,...,Inglês,Indicado,Atingiu PV,IPV,IAN,Fase ideal,Defas,Destaque IEG,Destaque IDA,Destaque IPV
0,RA-1,7,A,Aluno-1,2003,19,Menina,2016,Escola Pública,Ametista,...,6.0,Sim,Não,7.278,5.0,Fase 8 (Universitários),-1,Melhorar: Melhorar a sua entrega de lições de ...,Melhorar: Empenhar-se mais nas aulas e avaliaç...,Melhorar: Integrar-se mais aos Princípios Pass...
1,RA-2,7,A,Aluno-2,2005,17,Menina,2017,Rede Decisão,Ametista,...,9.7,Não,Não,6.778,10.0,Fase 7 (3º EM),0,Melhorar: Melhorar a sua entrega de lições de ...,Melhorar: Empenhar-se mais nas aulas e avaliaç...,Melhorar: Integrar-se mais aos Princípios Pass...
2,RA-3,7,A,Aluno-3,2005,17,Menina,2016,Rede Decisão,Ametista,...,6.9,Não,Não,7.556,10.0,Fase 7 (3º EM),0,Destaque: A sua boa entrega das lições de casa.,Melhorar: Empenhar-se mais nas aulas e avaliaç...,Destaque: A sua boa integração aos Princípios ...
3,RA-4,7,A,Aluno-4,2005,17,Menino,2017,Rede Decisão,Ametista,...,8.7,Não,Não,5.278,10.0,Fase 7 (3º EM),0,Melhorar: Melhorar a sua entrega de lições de ...,Melhorar: Empenhar-se mais nas aulas e avaliaç...,Melhorar: Integrar-se mais aos Princípios Pass...
4,RA-5,7,A,Aluno-5,2005,17,Menina,2016,Rede Decisão,Ametista,...,5.7,Não,Não,7.389,10.0,Fase 7 (3º EM),0,Destaque: A sua boa entrega das lições de casa.,Melhorar: Empenhar-se mais nas aulas e avaliaç...,Melhorar: Integrar-se mais aos Princípios Pass...


In [10]:
planilhas['PEDE2023'].info()

<class 'pandas.DataFrame'>
RangeIndex: 1014 entries, 0 to 1013
Data columns (total 48 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   RA                     1014 non-null   str    
 1   Fase                   1014 non-null   str    
 2   INDE 2023              931 non-null    float64
 3   Pedra 2023             931 non-null    str    
 4   Turma                  1014 non-null   str    
 5   Nome Anonimizado       1014 non-null   str    
 6   Data de Nasc           1014 non-null   object 
 7   Idade                  1014 non-null   object 
 8   Gênero                 1014 non-null   str    
 9   Ano ingresso           1014 non-null   int64  
 10  Instituição de ensino  1014 non-null   str    
 11  Pedra 20               240 non-null    str    
 12  Pedra 21               335 non-null    str    
 13  Pedra 22               600 non-null    str    
 14  Pedra 23               0 non-null      float64
 15  INDE 22        

In [11]:
planilhas['PEDE2023'].head()

,RA,Fase,INDE 2023,Pedra 2023,Turma,Nome Anonimizado,Data de Nasc,Idade,Gênero,Ano ingresso,...,Indicado,Atingiu PV,IPV,IAN,Fase Ideal,Defasagem,Destaque IEG,Destaque IDA,Destaque IPV,Destaque IPV.1
0,RA-861,ALFA,9.31095,Topázio,ALFA A - G0/G1,Aluno-861,6/17/2015,8,Feminino,2023,...,NaN,NaN,8.920,10.0,ALFA (1° e 2° ano),0,NaN,NaN,NaN,NaN
1,RA-862,ALFA,8.22120,Topázio,ALFA A - G0/G1,Aluno-862,5/31/2014,9,Masculino,2023,...,NaN,NaN,8.585,5.0,Fase 1 (3° e 4° ano),-1,NaN,NaN,NaN,NaN
2,RA-863,ALFA,5.92975,Quartzo,ALFA A - G0/G1,Aluno-863,2/25/2016,7,Masculino,2023,...,NaN,NaN,6.260,10.0,ALFA (1° e 2° ano),0,NaN,NaN,NaN,NaN
3,RA-864,ALFA,7.03400,Ametista,ALFA A - G0/G1,Aluno-864,2015-12-03 00:00:00,1900-01-08 00:00:00,Feminino,2023,...,NaN,NaN,8.500,10.0,ALFA (1° e 2° ano),0,NaN,NaN,NaN,NaN
4,RA-865,ALFA,8.15520,Topázio,ALFA A - G0/G1,Aluno-865,11/13/2014,8,Masculino,2023,...,NaN,NaN,7.915,10.0,ALFA (1° e 2° ano),0,NaN,NaN,NaN,NaN


In [12]:
planilhas['PEDE2024'].info()

<class 'pandas.DataFrame'>
RangeIndex: 1156 entries, 0 to 1155
Data columns (total 50 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   RA                     1156 non-null   str           
 1   Fase                   1156 non-null   object        
 2   INDE 2024              1092 non-null   object        
 3   Pedra 2024             1092 non-null   str           
 4   Turma                  1156 non-null   object        
 5   Nome Anonimizado       1156 non-null   str           
 6   Data de Nasc           1156 non-null   datetime64[us]
 7   Idade                  1156 non-null   int64         
 8   Gênero                 1156 non-null   str           
 9   Ano ingresso           1156 non-null   int64         
 10  Instituição de ensino  1155 non-null   str           
 11  Pedra 20               191 non-null    str           
 12  Pedra 21               264 non-null    str           
 13  Pedra 22      

In [13]:
planilhas['PEDE2024'].tail()

,RA,Fase,INDE 2024,Pedra 2024,Turma,Nome Anonimizado,Data de Nasc,Idade,Gênero,Ano ingresso,...,IPV,IAN,Fase Ideal,Defasagem,Destaque IEG,Destaque IDA,Destaque IPV,Escola,Ativo/ Inativo,Ativo/ Inativo.1
1151,RA-1658,9,INCLUIR,INCLUIR,9,Aluno-1658,2002-12-14 02:00:00,21,Masculino,2021,...,NaN,10.0,Fase 8 (Universitários),1,NaN,NaN,NaN,Faculdade (FIAP),Cursando,Cursando
1152,RA-1659,9,INCLUIR,INCLUIR,9,Aluno-1659,2003-02-04 02:00:00,21,Masculino,2021,...,NaN,10.0,Fase 8 (Universitários),1,NaN,NaN,NaN,Bolsista Universitário *Formado (a),Cursando,Cursando
1153,RA-1252,9,INCLUIR,INCLUIR,9,Aluno-1252,2002-06-03 03:00:00,22,Feminino,2021,...,NaN,10.0,Fase 8 (Universitários),1,NaN,NaN,NaN,Faculdade (FIAP),Cursando,Cursando
1154,RA-1660,9,INCLUIR,INCLUIR,9,Aluno-1660,2000-06-28 03:00:00,24,Feminino,2021,...,NaN,10.0,Fase 8 (Universitários),1,NaN,NaN,NaN,Bolsista Universitário *Formado (a),Cursando,Cursando
1155,RA-1661,9,INCLUIR,INCLUIR,9,Aluno-1661,2003-01-29 02:00:00,21,Feminino,2021,...,NaN,10.0,Fase 8 (Universitários),1,NaN,NaN,NaN,Bolsista Universitário *Formado (a),Cursando,Cursando


## 5. Conversão dos campos de data e idade

Conversão dos campos que apresentam formatos distintos entre os anos, padronizando:

- Ano de nascimento;
- Idade.

In [14]:
#Conversao Ano_2023
converter_campo(planilhas["PEDE2023"], "Data de Nasc", "ano")
converter_campo(planilhas["PEDE2023"], "Idade", "idade")

#Conversao Ano_2024
converter_campo(planilhas["PEDE2024"], "Data de Nasc", "ano")

## 6. Padronização dos nomes das colunas

Aplicação do dicionário de mapeamento para garantir que todas as planilhas possuam o mesmo esquema de atributos.

In [15]:
mapeamento_colunas = {
    # Identificação
    'Nome': 'Nome',
    'Nome Anonimizado': 'Nome',

    # Dados pessoais
    'Ano nasc': 'Ano_Nascimento',
    'Data de Nasc': 'Ano_Nascimento',
    'Idade 22': 'Idade',
    'Idade': 'Idade',

    # Indicadores
    'Defas': 'Defasagem',
    'Defasagem': 'Defasagem',

    'Fase ideal': 'Fase_Ideal',
    'Fase Ideal': 'Fase_Ideal',

    # Disciplinas
    'Inglês': 'Ing',
    'Ing': 'Ing',

    'Matem': 'Mat',
    'Mat': 'Mat',

    'Portug': 'Por',
    'Por': 'Por',

    'Gênero': 'Genero',
    'Ano ingresso': 'Ano_Ingresso',
    'Instituição de ensino': 'Instituicao_Ensino',
    'Nº Av': 'Num_Av',

    'Rec Av1': 'Rec_Av1',
    'Rec Av2': 'Rec_Av2',
    'Rec Av3': 'Rec_Av3',
    'Rec Av4': 'Rec_Av4',
    'Rec Av5': 'Rec_Av5',
    'Rec Av6': 'Rec_Av6',
    'Rec Psicologia': 'Rec_Psicologia',

    'Atingiu PV': 'Atingiu_PV',

    'Destaque IEG': 'Destaque_IEG',
    'Destaque IDA': 'Destaque_IDA',
    'Destaque IPV': 'Destaque_IPV',

    # Índices
    'INDE 20'  : 'INDE_20',
    'INDE 21'  : 'INDE_21',
    'INDE 22'  : 'INDE_22',
    'INDE 23'  : 'INDE_23_1',
    'INDE 2023': 'INDE_23',
    'INDE 2024': 'INDE_24',
    
    'Pedra 20'  : 'Pedra_20',
    'Pedra 21'  : 'Pedra_21',
    'Pedra 22'  : 'Pedra_22',
    'Pedra 23'  : 'Pedra_23_1',
    'Pedra 2023': 'Pedra_23',
    'Pedra 2024': 'Pedra_24',
    
    # Outros
    'Ativo/ Inativo': 'Ativo_Inativo',
    'Ativo/ Inativo.1': 'Ativo_Inativo_1',

    'Destaque IPV.1': 'Destaque_IPV_1',
}

for nome_aba, df in planilhas.items():
    df.rename(columns=mapeamento_colunas, inplace=True)

## 7. Validação após a padronização

Verificação da estrutura das três bases após a renomeação dos atributos.

In [16]:
planilhas['PEDE2022'].info()

<class 'pandas.DataFrame'>
RangeIndex: 860 entries, 0 to 859
Data columns (total 42 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   RA                  860 non-null    str    
 1   Fase                860 non-null    int64  
 2   Turma               860 non-null    str    
 3   Nome                860 non-null    str    
 4   Ano_Nascimento      860 non-null    int64  
 5   Idade               860 non-null    int64  
 6   Genero              860 non-null    str    
 7   Ano_Ingresso        860 non-null    int64  
 8   Instituicao_Ensino  860 non-null    str    
 9   Pedra_20            323 non-null    str    
 10  Pedra_21            462 non-null    str    
 11  Pedra_22            860 non-null    str    
 12  INDE_22             860 non-null    float64
 13  Cg                  860 non-null    int64  
 14  Cf                  860 non-null    int64  
 15  Ct                  860 non-null    int64  
 16  Num_Av             

In [17]:
planilhas['PEDE2023'].info()

<class 'pandas.DataFrame'>
RangeIndex: 1014 entries, 0 to 1013
Data columns (total 48 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   RA                  1014 non-null   str    
 1   Fase                1014 non-null   str    
 2   INDE_23             931 non-null    float64
 3   Pedra_23            931 non-null    str    
 4   Turma               1014 non-null   str    
 5   Nome                1014 non-null   str    
 6   Ano_Nascimento      1014 non-null   Int64  
 7   Idade               1014 non-null   Int64  
 8   Genero              1014 non-null   str    
 9   Ano_Ingresso        1014 non-null   int64  
 10  Instituicao_Ensino  1014 non-null   str    
 11  Pedra_20            240 non-null    str    
 12  Pedra_21            335 non-null    str    
 13  Pedra_22            600 non-null    str    
 14  Pedra_23_1          0 non-null      float64
 15  INDE_22             600 non-null    float64
 16  INDE_23_1        

In [18]:
planilhas['PEDE2024'].info()

<class 'pandas.DataFrame'>
RangeIndex: 1156 entries, 0 to 1155
Data columns (total 50 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   RA                  1156 non-null   str    
 1   Fase                1156 non-null   object 
 2   INDE_24             1092 non-null   object 
 3   Pedra_24            1092 non-null   str    
 4   Turma               1156 non-null   object 
 5   Nome                1156 non-null   str    
 6   Ano_Nascimento      1156 non-null   Int64  
 7   Idade               1156 non-null   int64  
 8   Genero              1156 non-null   str    
 9   Ano_Ingresso        1156 non-null   int64  
 10  Instituicao_Ensino  1155 non-null   str    
 11  Pedra_20            191 non-null    str    
 12  Pedra_21            264 non-null    str    
 13  Pedra_22            472 non-null    str    
 14  Pedra_23_1          690 non-null    str    
 15  INDE_22             472 non-null    float64
 16  INDE_23_1        

## 8. Integração das bases

Unificação das planilhas de 2022, 2023 e 2024 em um único DataFrame consolidado.

Também é criada a coluna **Ano_Referencia**, indicando o ano de origem de cada registro.

In [19]:
df_unificado = pd.concat(

    [

        df.assign(Ano=nome_aba)

        for nome_aba, df in planilhas.items()

    ],

    ignore_index=True,

    sort=False

)
# Eliminar colunas com menos de 10 valores não nulos
minimo_non_null = 10
df_unificado = remover_colunas_poucos_dados(df_unificado, minimo_non_null)

# Renomeia a coluna
df_unificado.rename(columns={'Ano': 'Ano_Referencia'}, inplace=True)

# Ordem desejada para as primeiras colunas
colunas_inicio = [
    'RA',
    'Fase',
    'Turma',
    'Nome',
    'Ano_Nascimento',
    'Idade',
    'Genero',
    'Ano_Ingresso',
    'Instituicao_Ensino',
    'Ano_Referencia'
]

# Demais colunas permanecem na ordem atual
outras_colunas = [
    col for col in df_unificado.columns
    if col not in colunas_inicio
]

# Reorganiza o DataFrame
df_unificado = df_unificado[colunas_inicio + outras_colunas]

2 colunas removidas:
['Destaque_IPV_1', 'Avaliador6']


## 9. Validação da base consolidada

Análise da base unificada para verificar:

- quantidade de registros;
- tipos dos atributos;
- distribuição dos anos de referência;
- consistência dos dados.

In [20]:
df_unificado.info()

<class 'pandas.DataFrame'>
RangeIndex: 3030 entries, 0 to 3029
Data columns (total 54 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   RA                  3030 non-null   str    
 1   Fase                3030 non-null   object 
 2   Turma               3030 non-null   object 
 3   Nome                3030 non-null   str    
 4   Ano_Nascimento      3030 non-null   Int64  
 5   Idade               3030 non-null   Int64  
 6   Genero              3030 non-null   str    
 7   Ano_Ingresso        3030 non-null   int64  
 8   Instituicao_Ensino  3029 non-null   str    
 9   Ano_Referencia      3030 non-null   str    
 10  Pedra_20            754 non-null    str    
 11  Pedra_21            1061 non-null   str    
 12  Pedra_22            1932 non-null   str    
 13  INDE_22             1932 non-null   float64
 14  Cg                  860 non-null    float64
 15  Cf                  860 non-null    float64
 16  Ct               

In [21]:
df_unificado['Ano_Referencia'].value_counts().head(25)


Ano_Referencia
PEDE2024    1156
PEDE2023    1014
PEDE2022     860
Name: count, dtype: int64

## 10. Análise de consistência

Avaliação da relação entre:

- Ano de nascimento;
- Idade;

com o objetivo de identificar possíveis inconsistências nos dados.

In [22]:
df_unificado['Idade'].value_counts().head(25)

Idade
11    367
12    364
10    357
9     306
13    298
14    278
15    226
8     214
16    187
17    153
18     67
7      61
19     43
20     37
21     35
22     21
23     10
24      3
26      1
27      1
25      1
Name: count, dtype: Int64

In [23]:
contagem =  df_unificado .groupby(['Ano_Nascimento','Idade']).size().reset_index(name="Quantidade")

print(contagem)

    Ano_Nascimento  Idade  Quantidade
0             1996     26           1
1             1996     27           1
2             1998     24           1
3             1998     25           1
4             1999     23           1
5             1999     24           1
6             2000     24           1
7             2001     21           2
8             2001     22           5
9             2001     23           9
10            2002     20           5
11            2002     21          13
12            2002     22          16
13            2003     19          10
14            2003     20          17
15            2003     21          20
16            2004     18          17
17            2004     19          17
18            2004     20          15
19            2005     17          56
20            2005     18          30
21            2005     19          16
22            2006     16          58
23            2006     17          46
24            2006     18          20
25          

## 11. Exportação da base tratada

Gravação da base consolidada em formato Excel (.xlsx), pronta para utilização nas próximas etapas do projeto.

In [24]:
salvar_excel(df_unificado, 'PEDE_Dados_Unificados.xlsx')

Arquivo salvo com sucesso: PEDE_Dados_Unificados.xlsx
